# LifelongReID Training on Kaggle

This notebook trains the LifelongReID model on IP102 dataset with lifelong learning metrics.

In [ ]:
import os
import sys
import subprocess

# Clone the repository
repo_url = os.environ.get('IP102_CODE_REPO', 'https://github.com/nta2112/LifelongReID-AKA-for-IP102')
repo_dir = '/kaggle/working/LifelongReID'

if not os.path.exists(repo_dir):
    print(f'Cloning from {repo_url}...')
    subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)
else:
    print('Repository already exists, pulling latest...')
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)

sys.path.insert(0, repo_dir)
os.chdir(repo_dir)
print(f'Working in: {os.getcwd()}')

In [ ]:
# Install requirements
!pip install -q -r requirements.txt 2>/dev/null || pip install -q torch torchvision numpy scipy scikit-learn matplotlib prettytable easydict Pillow

In [ ]:
# Verify dataset location
import glob

dataset_candidates = [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-dataset',
    '/kaggle/input/IP102',
    '/kaggle/input/ip102-dataset/IP102 dataset',
]

dataset_root = None
for cand in dataset_candidates:
    if os.path.exists(os.path.join(cand, 'train.json')):
        dataset_root = cand
        break

if dataset_root is None:
    # Deep walk to find it
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.json' in files and 'classes.txt' in files:
            dataset_root = root
            break

print(f'Dataset root: {dataset_root}')
print(f'Files: {os.listdir(dataset_root) if dataset_root else "Not found"}')

In [ ]:
# Import training modules
import torch
import argparse
from lreid.tools import time_now
from lreid.core import Base_metagraph_p_s
from lreid.data_loader import IncrementalReIDLoaders
from lreid.operation import train_p_s_an_epoch, fast_test_p_s
from lreid.evaluation import compute_all_metrics, MetricsLogger, compute_lifelong_metrics
from lreid.utils import get_num_gpus, setup_multi_gpu, get_loader_kwargs

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPUs: {torch.cuda.device_count()}')

In [ ]:
# Isolated LwFNet forward smoke test
import os
import time
import torch
from lreid.models.LwFnet import LwFNet

os.environ.pop('CUDA_LAUNCH_BLOCKING', None)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Forward test device: {device}')
if device.type != 'cuda':
    raise RuntimeError('CUDA is unavailable. Training on CPU is not practical for this model.')

smoke_model = LwFNet(class_num_list=[7, 6, 6, 6], pretrained=False).to(device)
smoke_model.train()
smoke_input = torch.randn(2, 3, 256, 128, device=device)
start = time.time()
with torch.no_grad():
    smoke_features, smoke_scores, _ = smoke_model(smoke_input, 0)
torch.cuda.synchronize()
print(f'LwFNet forward completed in {time.time() - start:.2f}s')
print(f'Feature shape: {tuple(smoke_features.shape)}, score shape: {tuple(smoke_scores.shape)}')
del smoke_model, smoke_input, smoke_features, smoke_scores
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
def run_train(model_name='res50', max_tasks=1, memory_size=2000,
              datasets_root=None, output_path=None, device='cuda',
              total_train_epochs=10, continual_step='5',
              p=16, k=4, steps=100):
    """
    Run training on IP102 dataset.
    
    Args:
        model_name: backbone model name
        max_tasks: number of tasks to run (0 = all tasks, 1 = quick test)
        memory_size: replay memory size (not used in this version)
        datasets_root: path to dataset root (auto-detect if None)
        output_path: path to save results
        device: cuda/cpu
        total_train_epochs: epochs per task
        continual_step: '5', '10', '1', or 'task'
        p: person count in a batch
        k: images per person in a batch
        steps: iterations per epoch
    """
    import time
    import os
    from lreid.utils.path_utils import find_dataset_root, get_task_split, load_ip102_class_mapping
    
    if datasets_root is None:
        datasets_root = find_dataset_root('IP102', env_var='IP102_DATA_ROOT')
    
    if output_path is None:
        output_path = f'results/{time.strftime("%Y-%m-%d-%H-%M-%S")}'
    
    os.makedirs(output_path, exist_ok=True)
    
    # Load class mappings
    class_id_to_name, valid_class_ids = load_ip102_class_mapping()
    task_splits = get_task_split(valid_class_ids)
    
    print(f'Found {len(valid_class_ids)} classes: {valid_class_ids}')
    print(f'Task splits: {[[len(t), t[:3]] for t in task_splits]}')
    
    # Determine number of tasks to run
    if max_tasks == 0:
        num_tasks = len(task_splits)
    else:
        num_tasks = min(max_tasks, len(task_splits))
    
    # Setup config
    config = argparse.Namespace()
    config.running_time = time.strftime('%Y-%m-%d-%H-%M-%S')
    config.output_path = output_path
    config.mode = 'train'
    config.continual_step = continual_step
    config.train_dataset = ['ip102']
    config.test_dataset = ['ip102']
    config.datasets_root = datasets_root
    config.combine_all = False
    config.image_size = [256, 128]
    config.test_batch_size = 64
    config.p = p
    config.k = k
    config.use_local_label4validation = True
    config.use_rea = True
    config.use_colorjitor = False
    config.cnnbackbone = model_name
    config.pid_num = len(valid_class_ids)
    config.steps = steps
    config.task_milestones = [25]
    config.task_gamma = 0.1
    config.new_module_milestones = [50, 100, 150, 200]
    config.new_module_gamma = 0.5
    config.task_base_learning_rate = 3.5e-4
    config.new_module_learning_rate = 3.5e-4
    config.weight_decay = 0.0005
    config.total_train_epochs = total_train_epochs
    config.total_continual_train_epochs = total_train_epochs
    config.epoch_start_joint = 80
    config.auto_resume_training_from_lastest_steps = True
    config.max_save_model_num = 2
    config.resume_train_dir = ''
    config.fast_test = True
    config.test_frequency = 5
    config.if_test_forget = True
    config.if_test_metagraph = False
    config.test_mode = 'all'
    config.test_metric = 'cosine'
    config.output_featuremaps = False
    config.save_heatmaps = False
    config.output_featuremaps_from_fixed = False
    config.weight_x = 1
    config.meta_graph_vertex_num = 32  # Reduced from 64 to avoid CUDA issues
    config.weight_r = 0.0005
    config.weight_t = 1
    config.t_margin = 0.3
    config.t_metric = 'euclidean'
    config.t_l2 = False
    config.weight_kd = 1
    config.kd_T = 2
    config.weight_fkd = 0
    config.fkd_l2 = False
    config.dropout = 0.6
    config.code_dim = 2048
    config.num_G_feature = 512
    config.fp_16 = False
    config.visualize_train_by_visdom = False
    config.re_init_lr_scheduler_per_step = False
    config.warmup_lr = False
    config.joint_train = False
    config.num_identities_per_domain = -1
    config.num_workers = 0
    
    # Setup multi-GPU
    import os
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # Better CUDA error messages
    num_gpus = get_num_gpus()
    print(f'Using {num_gpus} GPU(s)')
    
    # Reduce batch size to avoid OOM
    config.p = 8  # 8 persons per batch
    config.k = 2  # 2 images per person
    
    # Init loaders
    loaders = IncrementalReIDLoaders(config)
    
    # Update total_step based on actual tasks
    loaders.total_step = num_tasks
    
    # Init base model
    base = Base_metagraph_p_s(config, loaders)
    # Only wrap tasknet (backbone) in DataParallel - metagraph has torch.inverse() which fails with DataParallel
    base.model_dict['tasknet'], _ = setup_multi_gpu(base.model_dict['tasknet'])
    # metagraph stays on single GPU - ensure it's on the same device as tasknet output
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    base.model_dict['metagraph'] = base.model_dict['metagraph'].to(device)
    # Verify metagraph is on correct device
    for param in base.model_dict['metagraph'].parameters():
        assert param.device == device, f'Metagraph param on {param.device}, expected {device}'
    
    # Init logger
    from lreid.visualization import Logger
    logger = Logger(os.path.join(output_path, 'log.txt'))
    logger(config)
    
    # Init metrics logger
    metrics_logger = MetricsLogger(output_path)
    
    map_per_task = []
    
    for current_step in range(num_tasks):
        print(f'\n=== Task {current_step + 1}/{num_tasks} ===')
        
        if current_step > 0:
            logger(f'save_and_frozen old model in {current_step}')
            old_model = base.copy_model_and_frozen(model_name='tasknet')
            old_graph_model = base.copy_model_and_frozen(model_name='metagraph')
        else:
            old_model = None
            old_graph_model = None
        
        for current_epoch in range(total_train_epochs):
            try:
                # Train
                results = train_p_s_an_epoch(config, base, loaders, current_step, 
                                             old_model, old_graph_model, current_epoch, 
                                             output_featuremaps=False)
                results_dict, results_str = results
                logger(f'Time: {time_now()}; Step: {current_step}; Epoch: {current_epoch}; {results_str}')
            except RuntimeError as e:
                if 'CUDA' in str(e) or 'out of memory' in str(e).lower():
                    logger(f'CUDA Error at step {current_step}, epoch {current_epoch}: {e}')
                    torch.cuda.empty_cache()
                    # Try to continue with smaller batch
                    config.p = max(4, config.p // 2)
                    config.k = max(2, config.k // 2)
                    loaders = IncrementalReIDLoaders(config)
                    loaders.total_step = num_tasks
                    continue
                else:
                    raise
            
            if config.test_frequency > 0 and current_epoch % config.test_frequency == 0:
                rank_map_dict, rank_map_str = fast_test_p_s(config, base, loaders, current_step, 
                                                            if_test_forget=config.if_test_forget)
                logger(f'Time: {time_now()}; Test: {rank_map_str}')
        
        # Final test after task
        rank_map_dict, rank_map_str = fast_test_p_s(config, base, loaders, current_step,
                                                    if_test_forget=config.if_test_forget)
        logger(f'Time: {time_now()}; Step: {current_step} FINAL: {rank_map_str}')
        
        # Extract mAP for lifelong metrics
        task_map = rank_map_dict.get('ip102_fuse_mAP', rank_map_dict.get('ip102_tasknet_mAP', 0))
        map_per_task.append(task_map)
        
        # Compute lifelong metrics
        lifelong = compute_lifelong_metrics(map_per_task, [len(t) for t in task_splits[:current_step+1]])
        
        # Log to CSV/JSON
        metrics_logger.log_task(
            task_id=current_step + 1,
            num_classes=len(task_splits[current_step]),
            cnn_top1=rank_map_dict.get('ip102_tasknet_Rank1', 0),
            nme_top1=rank_map_dict.get('ip102_fuse_Rank1', 0),
            R1=rank_map_dict.get('ip102_tasknet_Rank1', 0),
            R5=0,  # Would need full evaluation
            R10=0,
            mAP=task_map,
            AUROC=lifelong.get('plasticity'),  # placeholder
            FPR95=lifelong.get('forgetting'),  # placeholder
            Plasticity=lifelong['plasticity'],
            Forgetting=lifelong['forgetting'],
            Overall=lifelong['overall']
        )
        
        if current_step > 0:
            del old_model, old_graph_model
    
    print(f'\nTraining complete! Results saved to {output_path}')
    print(f'Results CSV: {os.path.join(output_path, "results.csv")}')
    print(f'History JSON: {os.path.join(output_path, "history.json")}')
    
    return metrics_logger.get_history()

# Quick test run (uncomment to run)
# history = run_train(max_tasks=1, total_train_epochs=2, steps=10)
# print('History:', history)

# Run full 4-task training (max_tasks=0 means all tasks)
history = run_train(
    max_tasks=0,
    total_train_epochs=20,
    steps=150,
    datasets_root=dataset_root
)
print('Training history:', history)

In [ ]:
# CUDA and model forward smoke test
import os
import time
import torch

os.environ.pop('CUDA_LAUNCH_BLOCKING', None)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'CUDA device: {device}')
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    smoke_model = torch.nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3).to(device)
    smoke_input = torch.randn(2, 3, 256, 128, device=device)
    start = time.time()
    smoke_model(smoke_input)
    torch.cuda.synchronize()
    print(f'CUDA smoke test: {time.time() - start:.2f}s')
else:
    print('CUDA is unavailable; training would run on CPU and be extremely slow.')

# Display results
import pandas as pd

results_files = glob.glob('results/*/results.csv')
if results_files:
    latest = max(results_files, key=os.path.getmtime)
    df = pd.read_csv(latest)
    print(f'Results from: {latest}')
    print(df.to_string())
else:
    print('No results.csv found yet')